##  Assignement 1

## 1. Import the needed libraries

In [1]:
import sys #needed for vscode to recognize the topologicpy package
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.21) is OLDER than the latest version (0.9.33) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [3]:
renderer = "vscode"

## 4. Import the DXF files, Check them and Show them

In [4]:
# --- Outer boundary ---
raw_outer = Topology.ByDXFPath(r"C:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\Ext_Wire.dxf")
cluster_outer = Cluster.ByTopologies(raw_outer)
merged_outer = Topology.SelfMerge(cluster_outer)
print("Merged outer type:", type(merged_outer))

# merged_outer is already a Wire — use it directly as the external boundary
eb = merged_outer
print("External boundary:", eb)
print("Is closed:", Wire.IsClosed(eb))

# --- Inner boundaries ---
raw_inner = Topology.ByDXFPath(r"C:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\Int_Wire.dxf")
cluster_inner = Cluster.ByTopologies(raw_inner)
merged_inner = Topology.SelfMerge(cluster_inner)
inner_wires = Topology.Wires(merged_inner)
print("Inner wires:", len(inner_wires))

# --- Rebuild clean face ---
first_floor = Face.ByWires(eb, inner_wires)
print("first_floor:", first_floor)

Topology.Show(first_floor, renderer=renderer)


Merged outer type: <class 'topologic_core.Wire'>
External boundary: <topologic_core.Wire object at 0x000002666AC852F0>
Is closed: True
Inner wires: 12
first_floor: <topologic_core.Face object at 0x000002664D7546F0>


## 5. Manipulate Geometry Visualization

In [6]:
# Displaying the grouped geometry
Topology.Show(
    first_floor,
    faceColor=[210, 210, 250],
    faceOpacity=0.5,
    edgeColor="black",
    edgeWidth=3,
    showVertices=False,
    backgroundColor="white",
    width=800,
    height=600,
    renderer="vscode"

)

## 6. Definining Utility Functions

In [7]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


In [9]:
Topology.Show(first_floor,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [12]:
b_r = Wire.BoundingRectangle(first_floor)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+1))
vRange = list(range(0,int(length)+1))

grid = Grid.EdgesByDistances(first_floor, clip=True, uRange=uRange, vRange=vRange)

In [13]:
Topology.Show(first_floor, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 8. Slice the floor plan with the grid to create a topologic shell

In [15]:
shell = Topology.Slice(first_floor, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

In [16]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 9. Derive navigation and analysis graphs from the shell

In [17]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 10. Derive and store the analysis graph vertices

In [18]:
g_verts = Graph.Vertices(analysis_graph)

## 11. Show the analysis graph

In [19]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## 12. Spatial Intelligence through Graph Analysis

### a. Shortest Path (Use navigation graph)


In [21]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host= first_floor)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

    Topology.Show( first_floor, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Shortest Path Duration: 0.3 seconds
Straighten Wire Duration: 2.05 seconds
Original Shortest Path Length: 30.81
Straightened Shortened Path Length: 25.75


### b. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [22]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [23]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [25]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### c. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [26]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [27]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [28]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

In [29]:
status = Topology.ExportToBREP(first_floor, path=r"C:\Users\Win11\GraphML_RaniaChihaoui\Exports\first_floor.brep", overwrite=True)
print(status)

True
